# 02 Rebuild dataset from raw BDD100K zip files

This notebook rebuilds the current project dataset directly from the three raw archives in `EcoCAR/downloads` without relying on the old notebook2 pipeline.

In [ ]:
from pathlib import Path
import json, os, sys, shutil, tarfile

PROJECT_ROOT = Path('/content/drive/MyDrive/EcoCAR/DETR_GeoLane_pipeline')
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path('/content/DETR_GeoLane_pipeline')
assert PROJECT_ROOT.exists(), f'Project root not found: {PROJECT_ROOT}'
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

ECOCAR_ROOT = Path('/content/drive/MyDrive/EcoCAR')
DOWNLOADS = ECOCAR_ROOT / 'downloads'
DATASETS_DRIVE = ECOCAR_ROOT / 'datasets'

# Fast local build locations in the current runtime
RAW_ROOT = Path('/content/bdd100k_raw')
OUTPUT_ROOT = Path('/content/bdd100k_vehicle5')

# Persistent Drive locations that notebook00 can reuse later
DRIVE_OUTPUT_ROOT = DATASETS_DRIVE / 'bdd100k_vehicle5'
DRIVE_TAR_PATH = DATASETS_DRIVE / 'bdd100k_vehicle5.tar'

DATASETS_DRIVE.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT      =', PROJECT_ROOT)
print('DOWNLOADS         =', DOWNLOADS)
print('RAW_ROOT          =', RAW_ROOT)
print('OUTPUT_ROOT       =', OUTPUT_ROOT)
print('DRIVE_OUTPUT_ROOT =', DRIVE_OUTPUT_ROOT)
print('DRIVE_TAR_PATH    =', DRIVE_TAR_PATH)


In [ ]:
from src.data_prep import inspect_download_archives

archive_preview = inspect_download_archives(DOWNLOADS)
for name, members in archive_preview.items():
    print('\n' + '='*90)
    print(name)
    if not members:
        print('NOT FOUND')
    else:
        for m in members[:30]:
            print(m)

In [ ]:
from src.data_prep import rebuild_dualpath_dataset

summary = rebuild_dualpath_dataset(
    downloads_dir=DOWNLOADS,
    raw_root=RAW_ROOT,
    output_root=OUTPUT_ROOT,
    force_reextract=False,
)

print(json.dumps(summary, indent=2))

In [ ]:
from collections import Counter

def scan_raw_ids(label_dir, max_files=2000):
    counter = Counter()
    txts = sorted(label_dir.glob('*.txt'))[:max_files]
    for p in txts:
        for line in p.read_text().splitlines():
            s = line.strip().split()
            if len(s) >= 5:
                counter[int(float(s[0]))] += 1
    return counter

train_counter = scan_raw_ids(OUTPUT_ROOT / 'labels' / 'train', 3000)
val_counter = scan_raw_ids(OUTPUT_ROOT / 'labels' / 'val', 1000)
print('train raw ids ->', dict(sorted(train_counter.items())))
print('val raw ids   ->', dict(sorted(val_counter.items())))
print('expected ids  -> {0: car, 1: truck, 2: bus, 3: motorcycle, 4: bicycle}')

In [ ]:
yaml_path = OUTPUT_ROOT / 'bdd100k_vehicle5.yaml'
paths_cfg = OUTPUT_ROOT / 'paths_config.yaml'
print('Dataset YAML:', yaml_path)
print(yaml_path.read_text())
print('Paths config:', paths_cfg)
print(paths_cfg.read_text())

## 6 — Persist rebuilt dataset to Drive
Copy the rebuilt dataset to Drive and create a tar archive so `notebook00` can reuse it in a fresh runtime.

In [ ]:
# Remove previous Drive copy if requested, then sync the rebuilt dataset
assert OUTPUT_ROOT.exists(), f'Local output root missing: {OUTPUT_ROOT}'
if DRIVE_OUTPUT_ROOT.exists():
    print(f'Removing old Drive dataset: {DRIVE_OUTPUT_ROOT}')
    shutil.rmtree(DRIVE_OUTPUT_ROOT)

print(f'Copying dataset to Drive: {DRIVE_OUTPUT_ROOT}')
shutil.copytree(OUTPUT_ROOT, DRIVE_OUTPUT_ROOT)

print('Drive dataset ready:')
for sub in ['images/train', 'images/val', 'labels/train', 'labels/val']:
    p = DRIVE_OUTPUT_ROOT / sub
    print(f'  {sub}:', len(list(p.glob('*'))) if p.exists() else 'MISSING')
print('  yaml:', (DRIVE_OUTPUT_ROOT / 'bdd100k_vehicle5.yaml').exists())
print('  paths_config:', (DRIVE_OUTPUT_ROOT / 'paths_config.yaml').exists())


In [ ]:
# Create a tar archive in Drive for fast reuse in notebook00 or other runtimes
if DRIVE_TAR_PATH.exists():
    print(f'Removing old tar: {DRIVE_TAR_PATH}')
    DRIVE_TAR_PATH.unlink()

print(f'Creating tar archive: {DRIVE_TAR_PATH}')
with tarfile.open(DRIVE_TAR_PATH, 'w') as tar:
    tar.add(DRIVE_OUTPUT_ROOT, arcname='bdd100k_vehicle5', filter='data')

print('Tar created:', DRIVE_TAR_PATH)
print('Tar size (GB):', round(DRIVE_TAR_PATH.stat().st_size / (1024**3), 3))


In [ ]:
# Final summary for notebook00 reuse
final_summary = {
    'local_output_root': str(OUTPUT_ROOT),
    'drive_output_root': str(DRIVE_OUTPUT_ROOT),
    'drive_tar_path': str(DRIVE_TAR_PATH),
    'yaml_path': str(DRIVE_OUTPUT_ROOT / 'bdd100k_vehicle5.yaml'),
    'paths_config': str(DRIVE_OUTPUT_ROOT / 'paths_config.yaml'),
}
print(json.dumps(final_summary, indent=2))
